In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [19]:
# 1. Load Data
df = pd.read_csv('UCI_credit_card.csv')

df.head(10)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
5,6,50000.0,1,1,2,37,0,0,0,0,...,19394.0,19619.0,20024.0,2500.0,1815.0,657.0,1000.0,1000.0,800.0,0
6,7,500000.0,1,1,2,29,0,0,0,0,...,542653.0,483003.0,473944.0,55000.0,40000.0,38000.0,20239.0,13750.0,13770.0,0
7,8,100000.0,2,2,2,23,0,-1,-1,0,...,221.0,-159.0,567.0,380.0,601.0,0.0,581.0,1687.0,1542.0,0
8,9,140000.0,2,3,1,28,0,0,2,0,...,12211.0,11793.0,3719.0,3329.0,0.0,432.0,1000.0,1000.0,1000.0,0
9,10,20000.0,1,3,2,35,-2,-2,-2,-2,...,0.0,13007.0,13912.0,0.0,0.0,0.0,13007.0,1122.0,0.0,0


In [30]:
# Rename key columns for readability
df = df.rename(columns={
    'PAY_0': 'DELAY_MONTHS',
    'default.payment.next.month': 'DEFAULT'
})

In [31]:
# 2. Basic Feature Engineering 
# Utilization ratio: Clip negative bill amount to 0
df['UTILIZATION_RATE'] = (df['BILL_AMT1'].clip(lower=0) / df['LIMIT_BAL']).round(2)

In [32]:
# Assign mock branch regions for portfolio completeness
np.random.seed(42)
regions = ['North', 'South', 'East', 'West']
df['BRANCH_REGION'] = np.random.choice(regions, size=len(df))

In [33]:
# 3. Model Preparation (Logistic Regression)
features = ['LIMIT_BAL', 'AGE', 'DELAY_MONTHS', 'BILL_AMT1', 'PAY_AMT1', 'UTILIZATION_RATE']
x = df[features]
y = df['DEFAULT']

In [23]:
print(df.columns.tolist())

['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default.payment.next.month', 'UTILIZATION_RATE', 'BRANCH_REGION']


In [36]:
# Split data into Train (80%) and Test(20%)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)


In [37]:
# Train baseline logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [39]:
# Predict risk probabilities for testing accounts
y_pred = model.predict(x_test)
print(f"Model Accuracy: {round(accuracy_score(y_test, y_pred) * 100, 2)}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Accuracy: 80.75%

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.97      0.89      4687
           1       0.68      0.23      0.34      1313

    accuracy                           0.81      6000
   macro avg       0.75      0.60      0.61      6000
weighted avg       0.79      0.81      0.77      6000



In [40]:
# 4. Export Processed Dataset For Power BI
df['DEFAULT_PROBABILITY'] = model.predict_proba(x)[:, 1].round(4)
df.to_csv('credit_card_processed_bi.csv', index=False)
print("File exported: credit_card_processed_bi.csv")

File exported: credit_card_processed_bi.csv
